# LeetCode 3861 — Minimum Capacity Box bằng Python

Link đề: https://leetcode.com/problems/minimum-capacity-box/ · Độ khó: **Easy** · Chủ đề: Array, Enumeration, Argmin có điều kiện

Skeleton LeetCode:

```python
class Solution:
    def minimumIndex(self, capacity: List[int], itemSize: int) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Weekly Contest 492, câu 1, Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong các hộp **đủ chứa** món đồ (`capacity[i] >= itemSize`), chọn hộp có **sức chứa nhỏ nhất**; nếu hòa thì lấy **chỉ số nhỏ nhất**; không có hộp nào đủ thì trả `-1`.
2. Nắm mẫu bài "argmin có điều kiện lọc": tìm **chỉ số** của giá trị nhỏ nhất trong tập con thỏa điều kiện.
3. Biết dùng `enumerate`, so sánh tuple `(giá_trị, chỉ_số)`, `min(..., default=...)`, `list.index`.
4. Viết 5 cách khác nhau (đối chiếu từng cặp, một lượt duyệt, `min` trên tuple, hai lượt, quét theo giá trị) và so sánh bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: bỏ qua `itemSize`, dùng `<=` làm lệch tie-break, dùng `>` thay `>=`, trả về sức chứa thay vì chỉ số, quên `-1`.

## 2. Đề bài & ví dụ

Cho mảng `capacity`, trong đó `capacity[i]` là sức chứa của hộp thứ `i`, và một số nguyên `itemSize` là kích thước món đồ. Hộp `i` chứa được món đồ nếu `capacity[i] >= itemSize`.

Trả về **chỉ số của hộp có sức chứa nhỏ nhất** trong số các hộp chứa được món đồ. Nếu có nhiều hộp cùng sức chứa nhỏ nhất đó, trả về hộp có **chỉ số nhỏ nhất**. Nếu **không** có hộp nào chứa được, trả về `-1`.

| Ví dụ | capacity | itemSize | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[1,5,3,7]` | `3` | `2` | Hộp 1, 2, 3 đủ chỗ (5, 3, 7); nhỏ nhất là 3 ở chỉ số 2 |
| 2 | `[3,5,4,3]` | `2` | `0` | Cả bốn hộp đều đủ; nhỏ nhất là 3 xuất hiện ở chỉ số 0 và 3, lấy 0 |
| 3 | `[4]` | `5` | `-1` | Không hộp nào đủ chỗ |

Constraints: `1 <= capacity.length <= 100`; `1 <= capacity[i] <= 100`; `1 <= itemSize <= 100`.

In [1]:
for cap, size in [([1, 5, 3, 7], 3), ([3, 5, 4, 3], 2), ([4], 5)]:
    ok = [(i, c) for i, c in enumerate(cap) if c >= size]
    print(f"capacity={cap} itemSize={size}: hộp đủ chỗ (chỉ số, sức chứa) = {ok}")

capacity=[1, 5, 3, 7] itemSize=3: hộp đủ chỗ (chỉ số, sức chứa) = [(1, 5), (2, 3), (3, 7)]
capacity=[3, 5, 4, 3] itemSize=2: hộp đủ chỗ (chỉ số, sức chứa) = [(0, 3), (1, 5), (2, 4), (3, 3)]
capacity=[4] itemSize=5: hộp đủ chỗ (chỉ số, sức chứa) = []


## 3. Kiến thức nền

### 3.1. `enumerate`: lấy cả chỉ số và giá trị

`enumerate(list)` sinh các cặp `(chỉ_số, giá_trị)`. Dùng khi đề hỏi về **chỉ số**.

In [2]:
cap = [1, 5, 3, 7]
for i, c in enumerate(cap):
    print(i, c)

0 1
1 5
2 3
3 7


### 3.2. So sánh tuple (từ điển) và `min`

`(3, 2) < (3, 5)` vì phần tử đầu bằng nhau thì xét phần tử sau. `min` trên danh sách tuple `(sức_chứa, chỉ_số)` cho ra hộp có sức chứa nhỏ nhất, hòa thì chỉ số nhỏ nhất — chính xác yêu cầu của đề.

In [3]:
pairs = [(5, 1), (3, 2), (7, 3), (3, 0)]
print(min(pairs), (3, 0) < (3, 2), (3, 2) < (5, 1))

(3, 0) True True


### 3.3. `min(..., default=...)` với iterable rỗng

`min([])` gây `ValueError`. Truyền `default=` để trả giá trị mặc định khi rỗng: `min([], default=None)` là `None`.

In [4]:
print(min([4, 2, 9], default=None), min([], default=None), min((), default=-1))
try:
    min([])
except ValueError as e:
    print("ValueError:", e)

2 None -1
ValueError: min() iterable argument is empty


### 3.4. `list.index(x)`

`lst.index(x)` trả về chỉ số của **lần xuất hiện đầu tiên** của `x` (hoặc `ValueError` nếu không có). Với giá trị trùng nhau nó trả chỉ số **nhỏ nhất** — đúng tie-break của đề.

In [5]:
cap = [3, 5, 4, 3]
print(cap.index(3), cap.index(4))
print(3 in cap, 9 in cap)

0 2
True False


### 3.5. Giá trị vô cực làm "người canh" ban đầu

Để tìm giá trị nhỏ nhất ta khởi tạo `best = float('inf')` (lớn hơn mọi số) hoặc `best = 101` (vì `capacity[i] <= 100`). Sau khi duyệt, nếu `best` vẫn là giá trị ban đầu nghĩa là chưa có ứng viên.

In [6]:
best = float('inf')
print(best > 10**9, min(best, 5))
best = 101
print(any(c < best for c in [4, 7]))

True 5
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Bài toán hai bước: **(1) lọc** các hộp đủ chỗ; **(2) chọn** hộp nhỏ nhất, hòa thì chỉ số nhỏ.

Tính tay `capacity = [3, 5, 4, 3]`, `itemSize = 4`:

| chỉ số | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| sức chứa | 3 | 5 | 4 | 3 |
| `>= 4`? | không | có | có | không |

Các hộp đủ chỗ: chỉ số 1 (5) và chỉ số 2 (4). Nhỏ nhất là 4 -> chỉ số `2`.

Với `capacity = [3, 5, 4, 3]`, `itemSize = 2`: cả 4 hộp đủ chỗ, nhỏ nhất là 3 ở chỉ số 0 và 3 -> lấy `0` (chỉ số nhỏ hơn). Khi duyệt từ trái sang phải và chỉ cập nhật khi tìm được giá trị **nhỏ hơn hẳn** thì ta tự động giữ lại chỉ số đầu tiên.

In [7]:
def explain(cap, size):
    best_i, best_c = -1, None
    for i, c in enumerate(cap):
        fit = c >= size
        better = fit and (best_c is None or c < best_c)
        print(f"  i={i} c={c} đủ chỗ={fit} tốt hơn={better}")
        if better:
            best_i, best_c = i, c
    return best_i

print(explain([3, 5, 4, 3], 4))
print(explain([3, 5, 4, 3], 2))

  i=0 c=3 đủ chỗ=False tốt hơn=False
  i=1 c=5 đủ chỗ=True tốt hơn=True
  i=2 c=4 đủ chỗ=True tốt hơn=True
  i=3 c=3 đủ chỗ=False tốt hơn=False
2
  i=0 c=3 đủ chỗ=True tốt hơn=True
  i=1 c=5 đủ chỗ=True tốt hơn=False
  i=2 c=4 đủ chỗ=True tốt hơn=False
  i=3 c=3 đủ chỗ=True tốt hơn=False
0


## 5. Cách 1 — Brute Force / baseline

Đúng theo định nghĩa: hộp `i` là đáp án nếu (a) nó đủ chỗ và (b) **không có** hộp `j` nào đủ chỗ mà "tốt hơn" nó, tức `capacity[j] < capacity[i]` hoặc (`capacity[j] == capacity[i]` và `j < i`). Duyệt `i` tăng dần, trả về `i` đầu tiên thỏa.

Nút thắt: hai vòng lồng nhau `O(n²)`: mỗi hộp so với mọi hộp khác trong khi ta chỉ cần nhớ ứng viên tốt nhất hiện tại.

In [8]:
class SolutionBrute:
    def minimumIndex(self, capacity, itemSize):
        n = len(capacity)
        for i in range(n):
            if capacity[i] < itemSize:
                continue
            beaten = False
            for j in range(n):
                if capacity[j] >= itemSize and (capacity[j] < capacity[i] or (capacity[j] == capacity[i] and j < i)):
                    beaten = True
                    break
            if not beaten:
                return i
        return -1

print(SolutionBrute().minimumIndex([1, 5, 3, 7], 3), SolutionBrute().minimumIndex([3, 5, 4, 3], 2), SolutionBrute().minimumIndex([4], 5))

2 0 -1


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** duyệt **một lượt** từ trái sang phải, giữ `best_idx` (hộp tốt nhất hiện tại, ban đầu `-1`). Với hộp `i`:

- nếu `capacity[i] < itemSize`: bỏ qua;
- nếu chưa có ứng viên **hoặc** `capacity[i] < capacity[best_idx]` (**nhỏ hơn hẳn**): `best_idx = i`.

**Invariant:** sau khi xử lý `i` hộp đầu, `best_idx` là hộp có sức chứa nhỏ nhất trong các hộp đủ chỗ trong `capacity[:i]`, và nếu hòa thì là hộp có chỉ số nhỏ nhất. Vì ta chỉ đổi ứng viên khi **nhỏ hơn hẳn** (không phải `<=`) nên hộp đến sau với sức chứa bằng nhau không thay thế ứng viên cũ -> tie-break đúng.

Các cách tương đương: lấy `min` trên `(sức_chứa, chỉ_số)`; hoặc tìm sức chứa nhỏ nhất `m` rồi `capacity.index(m)`.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt duyệt với `best_idx`

Chính là thuật toán ở mục 6.

In [9]:
class SolutionOnePass:
    def minimumIndex(self, capacity, itemSize):
        best = -1
        for i, c in enumerate(capacity):
            if c >= itemSize and (best == -1 or c < capacity[best]):
                best = i
        return best

print(SolutionOnePass().minimumIndex([1, 5, 3, 7], 3), SolutionOnePass().minimumIndex([3, 5, 4, 3], 2), SolutionOnePass().minimumIndex([4], 5))

2 0 -1


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — `min` trên tuple `(sức_chứa, chỉ_số)`

Lọc các hộp đủ chỗ thành tuple `(c, i)` rồi lấy `min`; `default=None` để xử lý trường hợp không có hộp nào.

In [10]:
class SolutionMinTuple:
    def minimumIndex(self, capacity, itemSize):
        best = min(((c, i) for i, c in enumerate(capacity) if c >= itemSize), default=None)
        return -1 if best is None else best[1]

print(SolutionMinTuple().minimumIndex([1, 5, 3, 7], 3), SolutionMinTuple().minimumIndex([3, 5, 4, 3], 2), SolutionMinTuple().minimumIndex([4], 5))

2 0 -1


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (generator, không tạo list).

### Cách 4 — Hai lượt: tìm sức chứa nhỏ nhất rồi `index`

Lượt 1: `fits = [c for c in capacity if c >= itemSize]`; nếu rỗng trả `-1`. Lượt 2: `capacity.index(min(fits))` cho chỉ số xuất hiện đầu tiên (đúng tie-break).

In [11]:
class SolutionTwoPass:
    def minimumIndex(self, capacity, itemSize):
        fits = [c for c in capacity if c >= itemSize]
        if not fits:
            return -1
        return capacity.index(min(fits))

print(SolutionTwoPass().minimumIndex([1, 5, 3, 7], 3), SolutionTwoPass().minimumIndex([3, 5, 4, 3], 2), SolutionTwoPass().minimumIndex([4], 5))

2 0 -1


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `fits`.

### Cách 5 — Quét theo giá trị từ `itemSize` đến 100

Vì sức chứa chỉ nằm trong `1..100`, ta thử các giá trị `v = itemSize, itemSize+1, ..., 100`; giá trị **đầu tiên** có mặt trong `capacity` chính là sức chứa nhỏ nhất đủ chỗ, và `capacity.index(v)` cho chỉ số nhỏ nhất. Cách này dựa vào miền giá trị nhỏ.

In [12]:
class SolutionScanValue:
    def minimumIndex(self, capacity, itemSize):
        for v in range(itemSize, 101):
            if v in capacity:
                return capacity.index(v)
        return -1

print(SolutionScanValue().minimumIndex([1, 5, 3, 7], 3), SolutionScanValue().minimumIndex([3, 5, 4, 3], 2), SolutionScanValue().minimumIndex([4], 5))

2 0 -1


**Độ phức tạp:** Time `O(100 · n)` ở xấu nhất (mỗi `in`/`index` là `O(n)`); Space `O(1)`. Hợp khi miền giá trị nhỏ; không dùng được nếu giá trị lên tới `10^9`.

## 8. TRACE từng bước

Trace Cách 2 với `capacity = [3, 5, 4, 3]`, `itemSize = 4`:

| i | c | `c >= 4`? | best trước | `c < capacity[best]`? | best sau |
|---|---|---|---|---|---|
| 0 | 3 | không | -1 | | -1 |
| 1 | 5 | có | -1 | chưa có ứng viên | 1 |
| 2 | 4 | có | 1 | 4 < 5 có | 2 |
| 3 | 3 | không | 2 | | 2 |

In [13]:
def trace_one_pass(capacity, itemSize):
    best = -1
    for i, c in enumerate(capacity):
        before = best
        if c < itemSize:
            print(f"i={i} c={c}: không đủ chỗ (< {itemSize}) -> best={best}")
            continue
        if best == -1:
            best = i
            note = "chưa có ứng viên -> nhận"
        elif c < capacity[best]:
            best = i
            note = f"{c} < {capacity[best]} (nhỏ hơn hẳn) -> thay"
        else:
            note = f"{c} >= capacity[{best}]={capacity[best]} -> giữ"
        print(f"i={i} c={c}: {note}; best {before} -> {best}")
    return best

assert trace_one_pass([3, 5, 4, 3], 4) == 2
assert trace_one_pass([3, 5, 4, 3], 2) == 0
assert trace_one_pass([4], 5) == -1

i=0 c=3: không đủ chỗ (< 4) -> best=-1
i=1 c=5: chưa có ứng viên -> nhận; best -1 -> 1
i=2 c=4: 4 < 4 (nhỏ hơn hẳn) -> thay; best 1 -> 2
i=3 c=3: không đủ chỗ (< 4) -> best=2
i=0 c=3: chưa có ứng viên -> nhận; best -1 -> 0
i=1 c=5: 5 >= capacity[0]=3 -> giữ; best 0 -> 0
i=2 c=4: 4 >= capacity[0]=3 -> giữ; best 0 -> 0
i=3 c=3: 3 >= capacity[0]=3 -> giữ; best 0 -> 0
i=0 c=4: không đủ chỗ (< 5) -> best=-1


In [14]:
def trace_scan_value(capacity, itemSize):
    for v in range(itemSize, 101):
        print(f"thử sức chứa {v}: có trong mảng? {v in capacity}")
        if v in capacity:
            print(f"-> chỉ số đầu tiên: {capacity.index(v)}")
            return capacity.index(v)
        if v >= itemSize + 3:
            print("... (dừng in sớm)")
            break
    return -1

trace_scan_value([1, 5, 3, 7], 3)

thử sức chứa 3: có trong mảng? True
-> chỉ số đầu tiên: 2


2

## 9. Code nộp LeetCode

Dùng Cách 2 (một lượt, `O(n)` thời gian, `O(1)` bộ nhớ).

In [15]:
from typing import List, Optional


class Solution:
    def minimumIndex(self, capacity: List[int], itemSize: int) -> int:
        best = -1
        for i, c in enumerate(capacity):
            if c >= itemSize and (best == -1 or c < capacity[best]):
                best = i
        return best


print(Solution().minimumIndex([1, 5, 3, 7], 3), Solution().minimumIndex([3, 5, 4, 3], 2), Solution().minimumIndex([4], 5))

2 0 -1


## 10. Vì sao thuật toán đúng

1. **Chỉ xét hộp đủ chỗ.** Điều kiện `c >= itemSize` loại mọi hộp không hợp lệ, nên `best` luôn trỏ tới hộp hợp lệ hoặc là `-1`.
2. **Invariant "tốt nhất đến nay".** Sau mỗi bước, `best` là hộp hợp lệ có sức chứa nhỏ nhất trong phần đã duyệt. Khi gặp hộp hợp lệ mới có sức chứa **nhỏ hơn hẳn** `capacity[best]`, nó tốt hơn nên thay thế; nếu **bằng** hoặc **lớn hơn** thì không cải thiện được.
3. **Tie-break.** Hộp xuất hiện sau với sức chứa bằng `capacity[best]` không thay thế `best` (vì điều kiện là `<`), nên chỉ số nhỏ nhất được giữ lại.
4. **Không có hộp hợp lệ.** `best` giữ nguyên `-1`, đúng yêu cầu.
5. Các cách khác (`min` trên `(c, i)`, `index` của giá trị nhỏ nhất, quét theo giá trị) đều tính cùng "sức chứa nhỏ nhất rồi chỉ số nhỏ nhất".

## 11. Phân tích Time / Space Complexity

Gọi `n = len(capacity)`.

- Cách 1: `O(n²)` thời gian, `O(1)` bộ nhớ.
- Cách 2, 3: `O(n)` thời gian, `O(1)` bộ nhớ.
- Cách 4: `O(n)` thời gian, `O(n)` bộ nhớ cho list `fits`.
- Cách 5: `O(V · n)` với `V <= 100` là miền giá trị; `O(1)` bộ nhớ.

Không có sự khác biệt best / worst ở cách 2 và 3 (luôn duyệt hết mảng). Cách 5 có thể dừng sớm khi gặp giá trị ngay đầu miền.

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, các biên và 400 input ngẫu nhiên đúng constraints.

In [16]:
import random

solutions = [SolutionBrute(), SolutionOnePass(), SolutionMinTuple(), SolutionTwoPass(), SolutionScanValue(), Solution()]

fixed = [
    (([1, 5, 3, 7], 3), 2),
    (([3, 5, 4, 3], 2), 0),
    (([4], 5), -1),
    (([4], 4), 0),                  # bằng đúng itemSize vẫn chứa được
    (([5, 5, 5], 5), 0),
    (([9, 1, 9], 2), 0),
    (([2, 2, 100, 3], 3), 3),
    (([100, 100], 100), 0),
    (([1, 2, 3], 4), -1),
    (([7, 6, 6, 8], 6), 1),
]
for (cap, size), expected in fixed:
    for s in solutions:
        assert s.minimumIndex(list(cap), size) == expected, (type(s).__name__, cap, size)

random.seed(38610)
for _ in range(400):
    n = random.randint(1, 100)
    hi = random.choice([3, 10, 100])
    cap = [random.randint(1, hi) for _ in range(n)]
    size = random.randint(1, hi)
    outs = [s.minimumIndex(list(cap), size) for s in solutions]
    assert all(o == outs[0] for o in outs), (cap, size, outs)
    if outs[0] != -1:
        k = outs[0]
        assert cap[k] >= size and all(c < size or c >= cap[k] for c in cap)
        assert all(not (cap[j] == cap[k] and cap[j] >= size) for j in range(k))
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Nếu `itemSize` lớn hơn mọi sức chứa thì trả `-1`. (2) Nếu `itemSize = 1` thì đáp án là chỉ số của phần tử nhỏ nhất của cả mảng (vì mọi hộp đều đủ chỗ). (3) Thêm một hộp rất lớn (`100`) vào **cuối** không đổi đáp án khi đã có đáp án nhỏ hơn 100.

In [17]:
random.seed(3861)
for _ in range(300):
    cap = [random.randint(1, 99) for _ in range(random.randint(1, 60))]
    assert Solution().minimumIndex(cap, 100) == -1
    assert Solution().minimumIndex(cap, 1) == cap.index(min(cap))
    size = random.randint(1, 99)
    base = Solution().minimumIndex(cap, size)
    assert Solution().minimumIndex(cap + [100], size) == (base if base != -1 else len(cap))
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bỏ qua `itemSize` — lấy hộp nhỏ nhất của cả mảng

`capacity.index(min(capacity))` chọn hộp nhỏ nhất bất kể món đồ có vừa không. Với `[1, 5, 3, 7]`, `itemSize = 3` nhận `0` (hộp sức chứa 1 không vừa).

In [18]:
def wrong_ignore_size(capacity, itemSize):
    return capacity.index(min(capacity))                      # SAI: không lọc theo itemSize

print(wrong_ignore_size([1, 5, 3, 7], 3), "!=", Solution().minimumIndex([1, 5, 3, 7], 3))
assert wrong_ignore_size([1, 5, 3, 7], 3) == 0 and Solution().minimumIndex([1, 5, 3, 7], 3) == 2

0 != 2


### Lỗi 2: dùng `<=` làm lệch tie-break

Đổi `c < capacity[best]` thành `c <= capacity[best]` khiến hộp đến sau (chỉ số lớn) thay thế hộp trước khi sức chứa bằng nhau.

In [19]:
def wrong_le(capacity, itemSize):
    best = -1
    for i, c in enumerate(capacity):
        if c >= itemSize and (best == -1 or c <= capacity[best]):   # SAI: phải là <
            best = i
    return best

print(wrong_le([3, 5, 4, 3], 2), "!=", Solution().minimumIndex([3, 5, 4, 3], 2))
assert wrong_le([3, 5, 4, 3], 2) == 3 and Solution().minimumIndex([3, 5, 4, 3], 2) == 0

3 != 0


### Lỗi 3: dùng `>` thay vì `>=`

Hộp có sức chứa **đúng bằng** `itemSize` vẫn chứa được món đồ. Điều kiện `>` loại nhầm hộp đó.

In [20]:
def wrong_gt(capacity, itemSize):
    best = -1
    for i, c in enumerate(capacity):
        if c > itemSize and (best == -1 or c < capacity[best]):     # SAI: phải là >=
            best = i
    return best

print(wrong_gt([4], 4), "!=", Solution().minimumIndex([4], 4))
assert wrong_gt([4], 4) == -1 and Solution().minimumIndex([4], 4) == 0
assert wrong_gt([3, 5, 4, 3], 3) == 2 and Solution().minimumIndex([3, 5, 4, 3], 3) == 0

-1 != 0


### Lỗi 4: trả về sức chứa thay vì chỉ số

Đề hỏi **chỉ số** của hộp. Trả `min(fits)` là sai khi chỉ số và giá trị khác nhau.

In [21]:
def wrong_return_value(capacity, itemSize):
    fits = [c for c in capacity if c >= itemSize]
    return min(fits) if fits else -1                            # SAI: trả giá trị thay vì chỉ số

print(wrong_return_value([1, 5, 3, 7], 3), "!=", Solution().minimumIndex([1, 5, 3, 7], 3))
assert wrong_return_value([1, 5, 3, 7], 3) == 3 and Solution().minimumIndex([1, 5, 3, 7], 3) == 2

3 != 2


### Lỗi 5: khởi tạo `best = 0` (không phân biệt "chưa có ứng viên")

Nếu `best` bắt đầu bằng `0` mà hộp 0 không hợp lệ, ta so sánh nhầm với hộp 0 và có thể trả `0` khi thực ra không có đáp án.

In [22]:
def wrong_init_zero(capacity, itemSize):
    best = 0                                                    # SAI: -1 mới là 'chưa có'
    for i, c in enumerate(capacity):
        if c >= itemSize and c < capacity[best]:
            best = i
    return best

print(wrong_init_zero([4], 5), "!=", Solution().minimumIndex([4], 5))
assert wrong_init_zero([4], 5) == 0 and Solution().minimumIndex([4], 5) == -1
assert wrong_init_zero([1, 5, 3, 7], 3) == 0 and Solution().minimumIndex([1, 5, 3, 7], 3) == 2

0 != -1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chỉ số của giá trị nhỏ nhất thỏa điều kiện" -> **argmin có lọc**: một lượt, `best = -1`, cập nhật khi nhỏ hơn **hẳn**.
- Tie-break "chỉ số nhỏ nhất" -> duyệt xuôi và cập nhật với `<` (không `<=`); hoặc tuple `(giá_trị, chỉ_số)`.
- `min(generator, default=None)` xử lý gọn trường hợp rỗng.
- Luôn đọc kỹ `>=` hay `>`: ở đây đồ **vừa khít** vẫn vào được.
- Test bắt buộc: không có hộp nào đủ, nhiều hộp cùng sức chứa nhỏ nhất, hộp vừa khít, chỉ có một hộp.

## 15. Biến thể của bài

1. **Find Smallest Letter Greater Than Target** (744): phần tử nhỏ nhất lớn hơn mục tiêu.
2. **Search Insert Position** (35) / `bisect_left`: nếu mảng đã sắp xếp.
3. **Minimum Index Sum of Two Lists** (599): cực tiểu theo tổng chỉ số.
4. **Find the Index of the First Occurrence in a String** (28) và **Smallest Index With Equal Value** (2057): tìm chỉ số đầu tiên thỏa.
5. **Maximum Capacity**-style: tìm hộp **lớn nhất** không vượt quá một ngưỡng.
6. Biến thể: trả về **tất cả** chỉ số có sức chứa nhỏ nhất đủ chỗ; hoặc `k` hộp nhỏ nhất đủ chỗ (dùng `heapq`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | mỗi hộp so với mọi hộp khác | `O(n²)` | `O(1)` | Oracle, minh họa định nghĩa |
| 2. Một lượt | `best` + cập nhật khi nhỏ hơn hẳn | `O(n)` | `O(1)` | Nộp bài |
| 3. `min` tuple | `(c, i)` rồi lấy `min` | `O(n)` | `O(1)` | Ngắn gọn, ít lỗi tie-break |
| 4. Hai lượt | `min` giá trị + `index` | `O(n)` | `O(n)` | Dễ đọc |
| 5. Quét giá trị | thử `itemSize..100` | `O(V·n)` | `O(1)` | Khi miền giá trị nhỏ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minimumIndex([6, 2, 8, 2], 2)` và `Solution().minimumIndex([6, 2, 8, 2], 3)` ra gì?

**Bài 2 (tính tay):** `capacity = [10, 7, 7, 12]`, `itemSize = 8`: hộp nào được chọn?

**Bài 3 (điền TODO):** hoàn thành `maximum_index_not_exceeding(capacity, limit)` trả về chỉ số hộp có sức chứa **lớn nhất** không vượt quá `limit` (hòa thì chỉ số nhỏ nhất, không có thì `-1`).

**Bài 4 (tự viết lại):** viết `all_best_indices(capacity, itemSize)` trả về **mọi** chỉ số có sức chứa nhỏ nhất đủ chỗ (danh sách rỗng nếu không có).

**Bài 5 (mini challenge):** viết `k_smallest_fitting(capacity, itemSize, k)` trả về chỉ số của `k` hộp đủ chỗ có sức chứa nhỏ nhất (hòa thì chỉ số nhỏ trước), tối đa `k` phần tử.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minimumIndex([6, 2, 8, 2], 2), Solution().minimumIndex([6, 2, 8, 2], 3))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def maximum_index_not_exceeding_todo(capacity, limit):
    best = -1
    # TODO: lọc c <= limit, giữ giá trị lớn nhất (cập nhật khi lớn hơn hẳn)
    return best

1 0


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: lần 1 mọi hộp đều đủ chỗ (`itemSize = 2`); lần 2 hộp sức chứa 2 không đủ. Bài 2: hộp đủ chỗ là 10, 12 (7 < 8).

**Mức 2 (vừa):** Bài 3: giống Cách 2 nhưng điều kiện `c <= limit` và cập nhật khi `c > capacity[best]`. Bài 4: tìm `m = min(fits)` rồi lấy mọi `i` với `capacity[i] == m`. Bài 5: sắp xếp `sorted((c, i) ...)` rồi lấy `k` phần tử đầu.

**Mức 3 (gần đáp án):** Bài 1: `1` và `0`. Bài 2: hộp có sức chứa 10 -> chỉ số `0`. Bài 4: `all_best_indices([3,5,4,3], 2) = [0, 3]`. Bài 5: `k_smallest_fitting([3,5,4,3], 4, 2) = [2, 1]`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().minimumIndex([6, 2, 8, 2], 2) == 1 and Solution().minimumIndex([6, 2, 8, 2], 3) == 0
assert Solution().minimumIndex([10, 7, 7, 12], 8) == 0

# Bài 3
def maximum_index_not_exceeding(capacity, limit):
    best = -1
    for i, c in enumerate(capacity):
        if c <= limit and (best == -1 or c > capacity[best]):
            best = i
    return best
assert maximum_index_not_exceeding([3, 9, 5, 9], 9) == 1
assert maximum_index_not_exceeding([3, 9, 5], 2) == -1 and maximum_index_not_exceeding([3, 5, 4], 4) == 2

# Bài 4
def all_best_indices(capacity, itemSize):
    fits = [c for c in capacity if c >= itemSize]
    if not fits:
        return []
    m = min(fits)
    return [i for i, c in enumerate(capacity) if c == m]
assert all_best_indices([3, 5, 4, 3], 2) == [0, 3]
assert all_best_indices([4], 5) == [] and all_best_indices([1, 5, 3, 7], 3) == [2]

# Bài 5
def k_smallest_fitting(capacity, itemSize, k):
    pairs = sorted((c, i) for i, c in enumerate(capacity) if c >= itemSize)
    return [i for _, i in pairs[:k]]
assert k_smallest_fitting([3, 5, 4, 3], 4, 2) == [2, 1]
assert k_smallest_fitting([3, 5, 4, 3], 2, 3) == [0, 3, 2] and k_smallest_fitting([1], 2, 3) == []
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "chỉ số của giá trị nhỏ nhất thỏa điều kiện (hòa thì chỉ số nhỏ nhất)":**

```python
best = -1
for i, c in enumerate(arr):
    if điều_kiện(c) and (best == -1 or c < arr[best]):   # nhỏ hơn HẲN
        best = i
return best
# hoặc: min(((c, i) for i, c in enumerate(arr) if điều_kiện(c)), default=None)
```

**Checklist trước khi submit:**

- [ ] Điều kiện lọc là `capacity[i] >= itemSize` (vừa khít vẫn được)?
- [ ] Cập nhật bằng `<` (không `<=`) để giữ chỉ số nhỏ nhất khi hòa?
- [ ] `best` khởi tạo `-1` (không phải `0`) và trả `-1` khi không có hộp đủ chỗ?
- [ ] Trả về **chỉ số**, không phải sức chứa?
- [ ] Đã thử `[1,5,3,7]`/3, `[3,5,4,3]`/2, `[4]`/5?
- [ ] Chữ ký đúng `minimumIndex(self, capacity: List[int], itemSize: int) -> int`?